# Preprocessing and Augmentation
---

## Overview
Before feeding images into a neural network, we must prepare them properly.
Raw X-ray images come in different sizes, have pixel values ranging 0-255,
and our training set is limited in size. This task addresses all three problems.

## What we will do
1. Resize all images to a fixed shape (224x224)
2. Normalise pixel values from 0-255 to 0-1
3. Apply data augmentation to artificially expand the training set
4. Build Keras data generators for train, val and test splits

## Key Concept — Why Augmentation?
Our training set has only ~5,000 images. Deep learning models typically need
much more data to generalise well. Data augmentation creates new training
examples by applying random transformations — rotations, zooms, flips — to
existing images. This teaches the model to be robust to minor variations.

> **Important:** Augmentation is applied ONLY to training data.
> Val and test sets must reflect real-world conditions — no artificial changes.


---
## Step 1: Import Libraries
We import our custom `build_generators` function from `src/data_loader.py`
which encapsulates all the ImageDataGenerator logic in one reusable function.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
from src.data_loader import build_generators
from src.config import TRAIN_DIR, BATCH_SIZE
import matplotlib.pyplot as plt
import numpy as np
print('Libraries loaded successfully.')

---
## Step 2: Build Data Generators

`ImageDataGenerator` is Keras's built-in tool for:
- Loading images from folders in batches (memory efficient)
- Normalising pixel values on the fly
- Applying random augmentations during training

We use `.flow_from_directory()` which automatically reads the folder
structure and assigns labels based on subfolder names (NORMAL / PNEUMONIA).

**Augmentation parameters used:**
| Parameter | Value | Effect |
|-----------|-------|--------|
| rescale | 1./255 | Normalise pixels to 0-1 |
| rotation_range | 15 | Random rotation up to 15 degrees |
| zoom_range | 0.1 | Random zoom up to 10% |
| horizontal_flip | True | Random left-right flip |


In [ ]:
train_gen, val_gen, test_gen = build_generators()
print('Class indices:', train_gen.class_indices)
print(f'Train batches : {len(train_gen)}')
print(f'Val batches   : {len(val_gen)}')
print(f'Test batches  : {len(test_gen)}')

---
## Step 3: Inspect a Batch

We pull one batch from the training generator to verify:
- Image shape is (32, 224, 224, 3) — batch of 32 RGB images at 224x224
- Pixel values are in range [0, 1] after normalisation
- Labels are binary: 0 = NORMAL, 1 = PNEUMONIA

This is an important sanity check before training any model.


In [ ]:
batch_images, batch_labels = next(train_gen)
print(f'Batch shape  : {batch_images.shape}')
print(f'Pixel min    : {batch_images.min():.4f}')
print(f'Pixel max    : {batch_images.max():.4f}')
print(f'Label values : {sorted(set(batch_labels.astype(int)))}')

---
## Step 4: Visualise Augmented Training Batch

We plot a full batch of 24 augmented training images to visually confirm:
- Augmentation effects are visible (rotations, flips, zoom)
- Class distribution in the batch reflects training set imbalance
- Images look realistic — augmentation should not distort X-rays too aggressively

Each image title is colour coded: **green = NORMAL**, **red = PNEUMONIA**


In [ ]:
class_names = {0: 'NORMAL', 1: 'PNEUMONIA'}
fig, axes   = plt.subplots(3, 8, figsize=(20, 8))
fig.suptitle('Sample Augmented Training Batch', fontsize=14, fontweight='bold')

train_gen.reset()
images, labels = next(train_gen)

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        label = class_names[int(labels[i])]
        ax.set_title(label, fontsize=8,
                     color='green' if label == 'NORMAL' else 'red')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## Observations and Reflections

**1. What does rescale=1./255 do to the pixel values?**
rescale=1./255 normalizes pixel values from 0-255 to 0-1. This is a standard
preprocessing step for deep learning models, as it helps with faster convergence
during training and prevents larger pixel values from dominating the learning process.

**2. Why do we apply augmentation only to training data and not val/test?**
Augmentation is applied exclusively to training data to increase diversity and
reduce overfitting. Val and test sets are used to evaluate real-world performance
and must reflect the true data distribution without artificial transformations.
Augmenting them would give unrealistic performance estimates.

**3. What augmentation effects did you visually notice in the batch?**
- Rotation: Some images appear slightly rotated
- Zoom: Some images are zoomed in or out
- Horizontal Flip: Some images are horizontally flipped

**4. Why is shuffle=False important for val and test generators?**
- Reproducibility: Consistent image order across runs makes results reproducible
- Consistent Evaluation: Fixed order allows direct comparison of predictions
  with true labels, vital for error analysis and debugging. Shuffled data would
  make it impossible to track which prediction corresponds to which image.

---
## Summary

| Step | Detail |
|------|--------|
| Resize | All images resized to 224x224 |
| Normalise | Pixel values scaled to 0-1 via rescale=1./255 |
| Augmentation | Rotation 15deg, zoom 10%, horizontal flip |
| Generators | flow_from_directory loads batches on-the-fly |
| Val and Test | Only rescaled — no augmentation applied |

These generators are ready for model training in Tasks 3 and 5.
